In [3]:
# JUPYTER NOTEBOOK CELL: Team Development (Questions 43–45) using Logistic Regression

import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
import joblib
import os

# 1. Load the dataset (team.csv)
#    The CSV is assumed to have columns: "code", "job description", and "label"
file_path = "C:/Users/nosao/Desktop/Maxwell-Text Classification/Linear Response/data/linear_team_development.csv"
df = pd.read_csv(file_path, encoding='latin-1', on_bad_lines='skip')

# Normalize column names: lowercase and trimmed
df.columns = df.columns.str.strip().str.lower()
print("Available columns:", df.columns.tolist())

# Check that the expected column exists exactly as "job description"
if 'job description' not in df.columns or 'label' not in df.columns:
    raise ValueError("The dataset must have columns 'job description' and 'label'.")

# 2. Create separate label columns for Questions 43, 44, and 45
#    (Since the dataset has only one overall label, we duplicate it.)
df['question 43'] = df['label']
df['question 44'] = df['label']
df['question 45'] = df['label']

# Convert the new label columns to string type
for col in ['question 43', 'question 44', 'question 45']:
    df[col] = df[col].astype(str)

# 3. Extract features (X) and targets (y) for each question
X = df['job description'].astype(str)
y_q43 = df['question 43']
y_q44 = df['question 44']
y_q45 = df['question 45']

# 4. Split the data into train/test sets for each question (with stratification)
X_train_q43, X_test_q43, y_train_q43, y_test_q43 = train_test_split(
    X, y_q43, test_size=0.2, random_state=42, stratify=y_q43
)
X_train_q44, X_test_q44, y_train_q44, y_test_q44 = train_test_split(
    X, y_q44, test_size=0.2, random_state=42, stratify=y_q44
)
X_train_q45, X_test_q45, y_train_q45, y_test_q45 = train_test_split(
    X, y_q45, test_size=0.2, random_state=42, stratify=y_q45
)

# 5. Define a pipeline and hyperparameter grid using Logistic Regression
pipeline = make_pipeline(
    TfidfVectorizer(),
    LogisticRegression(max_iter=1000, class_weight='balanced')
)

param_grid = {
    'tfidfvectorizer__ngram_range': [(1,1), (1,2)],
    'tfidfvectorizer__max_df': [0.85, 0.95],
    'tfidfvectorizer__min_df': [1,5],
    'tfidfvectorizer__use_idf': [True, False],
    'logisticregression__C': [0.01, 0.1, 1, 10]
}

cv = StratifiedKFold(n_splits=5)

# 6. Train models for each question using GridSearchCV

# For Question 43
grid_q43 = GridSearchCV(pipeline, param_grid, cv=cv, scoring='accuracy', n_jobs=-1)
grid_q43.fit(X_train_q43, y_train_q43)
best_model_q43 = grid_q43.best_estimator_

# For Question 44
grid_q44 = GridSearchCV(pipeline, param_grid, cv=cv, scoring='accuracy', n_jobs=-1)
grid_q44.fit(X_train_q44, y_train_q44)
best_model_q44 = grid_q44.best_estimator_

# For Question 45
grid_q45 = GridSearchCV(pipeline, param_grid, cv=cv, scoring='accuracy', n_jobs=-1)
grid_q45.fit(X_train_q45, y_train_q45)
best_model_q45 = grid_q45.best_estimator_

# 7. Evaluate each model on its respective test set
def display_metrics(y_true, y_pred, question_num):
    print(f"Metrics for Question {question_num}:")
    print(classification_report(y_true, y_pred))
    print("Accuracy:", accuracy_score(y_true, y_pred))
    print()

y_pred_q43 = best_model_q43.predict(X_test_q43)
display_metrics(y_test_q43, y_pred_q43, 43)

y_pred_q44 = best_model_q44.predict(X_test_q44)
display_metrics(y_test_q44, y_pred_q44, 44)

y_pred_q45 = best_model_q45.predict(X_test_q45)
display_metrics(y_test_q45, y_pred_q45, 45)

# 8. Helper function to predict labels for a new job description
def predict_new_job_description(text):
    text_input = [text]  # wrap text in a list for vectorization
    pred_q43 = best_model_q43.predict(text_input)[0]
    pred_q44 = best_model_q44.predict(text_input)[0]
    pred_q45 = best_model_q45.predict(text_input)[0]
    print("Predicted labels for the new job description:")
    print(f"Question 43: {pred_q43}")
    print(f"Question 44: {pred_q44}")
    print(f"Question 45: {pred_q45}")

# Example usage with a new job description snippet
new_job_desc = """
Purpose of the Role: To provide an effective Joinery resource to ensure the University
fabric is efficiently maintained on a day-to-day basis including undertaking Project works. To
ensure the effective interaction of Estate and Facilities services with other services.

Responsible to: Estates Team Leader

Main Duties and Responsibilities:
1. To provide all forms of Joinery duties and tasks in which you are competent within the
University Estate possessing at least five years of trade experience. Working with the team
across various other construction trades.
2. To be responsible for day-to-day breakdown and reactive maintenance.
3. To participate in the Maintenance call-out rota team.
4. To be responsible for working to and delivering cyclical maintenance works ensuring
certain activities are carried out as per the PPM regime.
5. To manage the fire door programme focussing on the Inspection, maintenance (ART
accepted repair techniques), and repair to achieve a compliant campus. This will involve
a good theoretical knowledge of the standards and regulations.
6. To be responsible for the Fire door dashboard, upkeep, and maintenance of the system.
To provide information on defects and repair (analyzing reports), accepted repair
techniques, and input for Projects.
7. To maintain Fire door records on CAFM and in line with the Fire Safety Regulations
(2023). Records must be kept.
8. Identify hazards, defects, and the need for adjustment or repair; to ensure compliance
with agreed codes, law, working practices, and health and safety whilst carrying out your
duties.
9. To provide support and guidance to Contractors engaged in Fire door campus works and
act as a focal point ensuring a fully compliant Fire door install is delivered to the Estate.
10. To manage quantities required to complete each task and manage material stocks and
ordering process.
11. To be responsible for ensuring all tools and equipment are maintained in good working
order and ready for use including power tools within the Estate.
12. To ensure all University fixtures, fittings, furniture, doors, locks, flooring, and other
Joinery items are efficiently maintained, repaired, constructed, or replaced, working
closely with Estates, Facilities Managers, and Estates Team Leader to achieve.
13. To ensure works are delivered in compliance with documented risk assessments and Method
statements, and responsible for the production and review of role-specific risk
assessments.
14. To be responsible for a high standard of conduct always working in a safe and
professional manner reporting any health and safety-related issues to the Estates Team
Leader immediately.
15. To be responsible for working to and delivering all works and repairs in a manner that
ensures VFM and quality finishes are implemented and maintained.
16. To assist the team and organization with general duties over and above your core skills.
Promote, develop and expand the business of our organization generally meeting set
targets.
17. To aid and advise the other members of the University Estates & Facilities staff including
porters and grounds staff as required.
18. To adhere to all organization policies and procedures.
19. To be responsible for continued professional development ensuring the post holder is
conversant and aware of current regulations, legislation, and approved industry
standards to their job role. You will have a basic awareness of Asbestos, CDM,
health and safety regulations.
20. To undertake small projects as reasonably required of the job role and advise all Estates
teams to deliver solutions and cost reductions to all joinery works on campus.

General Duties:
21. To ensure the use of data complies with current regulations, particularly those relating to
GDPR.
22. To comply with all health, safety, and wellbeing policies and procedures at all times and to
take responsibility for promoting and safeguarding the welfare and protection of others.
23. To advocate, promote, and advance equity and social justice within your work.
24. To carry out other duties, commensurate with the grade of the post, as may reasonably be
directed by your line manager after due consultation.
"""
predict_new_job_description(new_job_desc)


# Create a directory to save models
save_dir = "saved_models"
os.makedirs(save_dir, exist_ok=True)  # Create the directory if it doesn't exist

# Save each best model separately
joblib.dump(best_model_q43, os.path.join(save_dir, "best_model_q43.pkl"))
joblib.dump(best_model_q44, os.path.join(save_dir, "best_model_q44.pkl"))
joblib.dump(best_model_q45, os.path.join(save_dir, "best_model_q45.pkl"))

print("Models saved successfully!")



Available columns: ['code', 'job description', 'label']
Metrics for Question 43:
              precision    recall  f1-score   support

  Response A       0.82      0.86      0.84        21
  Response B       0.75      0.75      0.75        16
  Response C       0.75      0.60      0.67         5

    accuracy                           0.79        42
   macro avg       0.77      0.74      0.75        42
weighted avg       0.78      0.79      0.78        42

Accuracy: 0.7857142857142857

Metrics for Question 44:
              precision    recall  f1-score   support

  Response A       0.82      0.86      0.84        21
  Response B       0.75      0.75      0.75        16
  Response C       0.75      0.60      0.67         5

    accuracy                           0.79        42
   macro avg       0.77      0.74      0.75        42
weighted avg       0.78      0.79      0.78        42

Accuracy: 0.7857142857142857

Metrics for Question 45:
              precision    recall  f1-score   s

In [ ]:
import joblib
import os


# Load the saved models
best_model_q43 = joblib.load(os.path.join(save_dir, "best_model_q43.pkl"))
best_model_q44 = joblib.load(os.path.join(save_dir, "best_model_q44.pkl"))
best_model_q45 = joblib.load(os.path.join(save_dir, "best_model_q45.pkl"))

print("Models loaded successfully!")


NameError: name 'joblib' is not defined

In [ ]:
# Example usage with a new job description snippet
new_job_desc = """

"""
predict_new_job_description(new_job_desc)


Predicted labels for the new job description:
Question 43: Response B
Question 44: Response B
Question 45: Response B
